## Example 1: Basic package import and trace encoder functionality

In [ ]:
# Example 1: Basic package import and trace encoder functionality
import opentelemetry.exporter.otlp.proto.common as proto_common
from opentelemetry.exporter.otlp.proto.common.trace_encoder import encode_spans
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

print("Example 1 - Basic package import and trace encoder functionality")

try:
    print(f"Package: {proto_common.__name__}")

    # Record finished spans in memory with a local tracer provider
    span_store = InMemorySpanExporter()
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(span_store))
    tracer = provider.get_tracer("test_tracer", "1.0")

    with tracer.start_as_current_span("example_operation") as span:
        span.set_attribute("operation.name", "basic_test")
        span.set_attribute("service.version", "1.0.0")
        span.add_event("Processing started")
        span.add_event("Processing completed")

    spans = span_store.get_finished_spans()
    assert len(spans) == 1, spans

    # encode_spans() builds the OTLP ExportTraceServiceRequest protobuf
    req = encode_spans(spans)
    assert len(req.resource_spans) == 1, req
    scope_spans = req.resource_spans[0].scope_spans[0]
    assert scope_spans.scope.name == "test_tracer", scope_spans.scope
    encoded = scope_spans.spans[0]
    assert encoded.name == "example_operation", encoded.name
    attrs = {kv.key: kv.value.string_value for kv in encoded.attributes}
    assert attrs == {"operation.name": "basic_test", "service.version": "1.0.0"}, attrs
    assert [ev.name for ev in encoded.events] == ["Processing started", "Processing completed"]
    assert encoded.trace_id == spans[0].context.trace_id.to_bytes(16, "big")
    payload = req.SerializeToString()
    assert len(payload) > 0
    print(f"Encoded span {encoded.name!r}: {len(attrs)} attributes, {len(encoded.events)} events, {len(payload)} bytes")

    print("Example 1 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_exporter_otlp_proto_common example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Metrics encoder functionality and testing

In [ ]:
# Example 2: Metrics encoder functionality and testing
from opentelemetry.exporter.otlp.proto.common.metrics_encoder import encode_metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.resources import Resource

print("Example 2 - Metrics encoder functionality and testing")

try:
    # Collect metrics in memory with a local meter provider
    reader = InMemoryMetricReader()
    resource = Resource.create({"service.name": "test-service", "service.version": "1.0.0"})
    provider = MeterProvider(resource=resource, metric_readers=[reader])
    meter = provider.get_meter("test_meter", version="1.0")

    request_counter = meter.create_counter("http_requests_total", description="Total number of HTTP requests", unit="1")
    request_duration = meter.create_histogram("http_request_duration_seconds", description="HTTP request duration in seconds", unit="s")
    active_connections = meter.create_up_down_counter("active_connections", description="Number of active connections", unit="1")

    durations = [0.1, 0.25, 0.5, 1.0, 2.0]
    for duration in durations:
        request_counter.add(1, {"method": "GET", "endpoint": "/api/data"})
        request_counter.add(1, {"method": "POST", "endpoint": "/api/submit"})
        request_duration.record(duration, {"method": "GET", "status": "200"})
    active_connections.add(3)
    active_connections.add(-1)
    print(f"Recorded {len(durations)} batches of counter, histogram and up-down counter values")

    # encode_metrics() builds the OTLP ExportMetricsServiceRequest protobuf
    req = encode_metrics(reader.get_metrics_data())
    assert len(req.resource_metrics) == 1, req
    resource_metrics = req.resource_metrics[0]
    resource_attrs = {kv.key: kv.value.string_value for kv in resource_metrics.resource.attributes}
    assert resource_attrs["service.name"] == "test-service", resource_attrs
    scope_metrics = resource_metrics.scope_metrics[0]
    assert scope_metrics.scope.name == "test_meter", scope_metrics.scope
    by_name = {m.name: m for m in scope_metrics.metrics}
    assert set(by_name) == {"http_requests_total", "http_request_duration_seconds", "active_connections"}, set(by_name)

    counter_points = by_name["http_requests_total"].sum.data_points
    assert len(counter_points) == 2 and sum(p.as_int for p in counter_points) == 10, counter_points
    assert by_name["http_requests_total"].sum.is_monotonic
    hist_point = by_name["http_request_duration_seconds"].histogram.data_points[0]
    assert hist_point.count == len(durations) and abs(hist_point.sum - sum(durations)) < 1e-9, hist_point
    assert by_name["active_connections"].sum.data_points[0].as_int == 2
    assert not by_name["active_connections"].sum.is_monotonic
    print(f"Encoded metrics: {sorted(by_name)}; request total = 10, histogram count = {hist_point.count}")

    print("Example 2 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_exporter_otlp_proto_common example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Advanced trace encoding with span hierarchy

In [ ]:
# Example 3: Advanced trace encoding with span hierarchy
from opentelemetry.exporter.otlp.proto.common.trace_encoder import encode_spans
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

print("Example 3 - Advanced trace encoding with span hierarchy")

try:
    resource = Resource.create({
        "service.name": "payment-service",
        "service.version": "2.1.0",
        "deployment.environment": "production"
    })
    span_store = InMemorySpanExporter()
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(SimpleSpanProcessor(span_store))
    tracer = provider.get_tracer("payment_processor", "2.1.0")

    # Parent span with one child span
    with tracer.start_as_current_span("process_payment") as parent_span:
        parent_span.set_attribute("payment.amount", 99.99)
        parent_span.add_event("Payment processing started")
        with tracer.start_as_current_span("validate_payment") as child_span:
            child_span.set_attribute("validation.type", "card_details")
            child_span.add_event("Validation completed")
        parent_span.set_status(Status(StatusCode.OK))

    req = encode_spans(span_store.get_finished_spans())
    resource_spans = req.resource_spans[0]
    resource_attrs = {kv.key: kv.value.string_value for kv in resource_spans.resource.attributes}
    assert resource_attrs["service.name"] == "payment-service", resource_attrs
    encoded = {s.name: s for s in resource_spans.scope_spans[0].spans}
    parent, child = encoded["process_payment"], encoded["validate_payment"]

    # The hierarchy survives encoding: same trace, child points at the parent
    assert child.trace_id == parent.trace_id
    assert child.parent_span_id == parent.span_id
    assert parent.parent_span_id == b""
    assert parent.status.code == parent.status.STATUS_CODE_OK, parent.status
    assert parent.attributes[0].key == "payment.amount" and parent.attributes[0].value.double_value == 99.99
    print(f"Encoded {len(encoded)} spans; child parent_span_id = {child.parent_span_id.hex()}")

    print("Example 3 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_exporter_otlp_proto_common example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Encoder integration with traces and metrics

In [ ]:
# Example 4: Encoder integration with traces and metrics
from opentelemetry.exporter.otlp.proto.common.metrics_encoder import encode_metrics
from opentelemetry.exporter.otlp.proto.common.trace_encoder import encode_spans
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

print("Example 4 - Encoder integration with traces and metrics")

try:
    # One resource shared by the tracer and the meter provider
    resource = Resource.create({
        "service.name": "ecommerce-api",
        "service.version": "3.0.1"
    })
    span_store = InMemorySpanExporter()
    tracer_provider = TracerProvider(resource=resource)
    tracer_provider.add_span_processor(SimpleSpanProcessor(span_store))
    reader = InMemoryMetricReader()
    meter_provider = MeterProvider(resource=resource, metric_readers=[reader])

    tracer = tracer_provider.get_tracer("ecommerce_tracer", "3.0.1")
    meter = meter_provider.get_meter("ecommerce_meter", version="3.0.1")
    order_counter = meter.create_counter("orders_total")
    order_value = meter.create_histogram("order_value")

    with tracer.start_as_current_span("process_order") as span:
        span.set_attribute("order.id", "ord_001")
        order_counter.add(1, {"source": "web"})
        order_value.record(150.00)

    trace_req = encode_spans(span_store.get_finished_spans())
    metrics_req = encode_metrics(reader.get_metrics_data())

    # Both requests carry the same resource
    def resource_attrs(resource_proto):
        return {kv.key: kv.value.string_value for kv in resource_proto.attributes}

    trace_resource = resource_attrs(trace_req.resource_spans[0].resource)
    metrics_resource = resource_attrs(metrics_req.resource_metrics[0].resource)
    assert trace_resource == metrics_resource, (trace_resource, metrics_resource)
    assert trace_resource["service.name"] == "ecommerce-api", trace_resource

    assert trace_req.resource_spans[0].scope_spans[0].spans[0].name == "process_order"
    metrics = {m.name: m for m in metrics_req.resource_metrics[0].scope_metrics[0].metrics}
    assert metrics["orders_total"].sum.data_points[0].as_int == 1
    assert metrics["order_value"].histogram.data_points[0].sum == 150.0
    print(f"Trace and metrics requests share resource service.name={trace_resource['service.name']!r}")

    print("Example 4 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_exporter_otlp_proto_common example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Error spans and a serialize/parse round trip

In [ ]:
# Example 5: Error spans and a serialize/parse round trip
from opentelemetry.exporter.otlp.proto.common.metrics_encoder import encode_metrics
from opentelemetry.exporter.otlp.proto.common.trace_encoder import encode_spans
from opentelemetry.proto.collector.metrics.v1.metrics_service_pb2 import ExportMetricsServiceRequest
from opentelemetry.proto.collector.trace.v1.trace_service_pb2 import ExportTraceServiceRequest
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

print("Example 5 - Error spans and a serialize/parse round trip")

try:
    resource = Resource.create({
        "service.name": "auth-service",
        "service.version": "4.2.1",
        "deployment.environment": "production"
    })
    span_store = InMemorySpanExporter()
    tracer_provider = TracerProvider(resource=resource)
    tracer_provider.add_span_processor(SimpleSpanProcessor(span_store))
    reader = InMemoryMetricReader()
    meter_provider = MeterProvider(resource=resource, metric_readers=[reader])

    tracer = tracer_provider.get_tracer("auth_service", "4.2.1")
    login_attempts = meter_provider.get_meter("auth_metrics", version="4.2.1").create_counter("login_attempts_total")

    # A failed login: the exception is recorded on the span and the status set to ERROR
    with tracer.start_as_current_span("authenticate_user") as auth_span:
        auth_span.set_attribute("user.id", "user123")
        try:
            raise ValueError("invalid credentials")
        except ValueError as exc:
            auth_span.record_exception(exc)
            auth_span.set_status(Status(StatusCode.ERROR, str(exc)))
            login_attempts.add(1, {"result": "failure"})
    login_attempts.add(2, {"result": "success"})

    trace_req = encode_spans(span_store.get_finished_spans())
    span = trace_req.resource_spans[0].scope_spans[0].spans[0]
    assert span.status.code == span.status.STATUS_CODE_ERROR, span.status
    assert span.status.message == "invalid credentials", span.status
    exception_event = span.events[0]
    assert exception_event.name == "exception", exception_event
    event_attrs = {kv.key: kv.value.string_value for kv in exception_event.attributes}
    assert event_attrs["exception.type"] == "ValueError", event_attrs
    assert event_attrs["exception.message"] == "invalid credentials", event_attrs

    # The wire bytes parse back into an identical request
    trace_bytes = trace_req.SerializeToString()
    assert ExportTraceServiceRequest.FromString(trace_bytes) == trace_req
    metrics_req = encode_metrics(reader.get_metrics_data())
    metrics_bytes = metrics_req.SerializeToString()
    assert ExportMetricsServiceRequest.FromString(metrics_bytes) == metrics_req
    points = metrics_req.resource_metrics[0].scope_metrics[0].metrics[0].sum.data_points
    by_result = {p.attributes[0].value.string_value: p.as_int for p in points}
    assert by_result == {"failure": 1, "success": 2}, by_result
    print(f"Round-tripped {len(trace_bytes)} trace bytes and {len(metrics_bytes)} metrics bytes")

    print("Example 5 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"opentelemetry_exporter_otlp_proto_common example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")